# Notebook 01: Catalog Exploration

**Objective:** Understand the OpenMetadata catalog structure, navigate metadata, and add a PostgreSQL data source.

**Prerequisites:**
- Catalog lab is running (`docker compose up -d` from `labs/catalog/`)
- OpenMetadata UI is accessible at http://localhost:18585
- Wait 3-5 minutes after starting for OpenMetadata to fully initialize

**What you will learn:**
1. Connect to the OpenMetadata API and check server health
2. List database services registered in the catalog
3. Add the lab PostgreSQL as a new database service via the UI
4. Discover tables via the API after metadata ingestion
5. Inspect column metadata for the `customers` table

## Setup: Configuration and Authentication

All notebooks share these connection settings. The OpenMetadata server runs inside Docker on port 8585 (internal), exposed on port 18585 (host). From within the Jupyter container, we connect via the Docker network using the service name `openmetadata-server`.

In [ ]:
import requests
import json
import time

# OpenMetadata API base URL (internal Docker network)
OM_URL = "http://openmetadata-server:8585/api/v1"

# Default admin credentials for lab use
OM_ADMIN_USER = "admin"
OM_ADMIN_PASSWORD = "admin"

print(f"OpenMetadata API: {OM_URL}")
print("Authenticating...")

### Authenticate with OpenMetadata

OpenMetadata uses JWT tokens for API authentication. We log in with the default admin credentials and store the token for subsequent API calls.

In [ ]:
def get_auth_token(base_url, username, password, max_retries=12, wait_seconds=15):
    """Authenticate with OpenMetadata and return JWT token.
    
    Retries if server is still starting up (can take 3-5 minutes).
    """
    login_url = f"{base_url}/users/login"
    payload = {"email": username, "password": password}
    
    for attempt in range(1, max_retries + 1):
        try:
            resp = requests.post(login_url, json=payload, timeout=10)
            if resp.status_code == 200:
                token = resp.json().get("accessToken") or resp.json().get("tokenType", "")
                if token:
                    print(f"Authenticated successfully (attempt {attempt})")
                    return resp.json()["accessToken"]
                # Some OM versions return token differently
                print(f"Login response: {resp.json().keys()}")
                return resp.json().get("accessToken", resp.text)
            else:
                print(f"Attempt {attempt}/{max_retries}: HTTP {resp.status_code} - retrying in {wait_seconds}s...")
        except requests.ConnectionError:
            print(f"Attempt {attempt}/{max_retries}: Server not ready - retrying in {wait_seconds}s...")
        except requests.Timeout:
            print(f"Attempt {attempt}/{max_retries}: Timeout - retrying in {wait_seconds}s...")
        
        time.sleep(wait_seconds)
    
    raise ConnectionError(
        f"Could not connect to OpenMetadata after {max_retries} attempts.\n"
        f"Check: docker compose logs openmetadata-server"
    )

# Authenticate and set up headers for all subsequent requests
token = get_auth_token(OM_URL, OM_ADMIN_USER, OM_ADMIN_PASSWORD)
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
print(f"Token: {token[:20]}...")

## Exercise 1: Check Server Health and Version

Let's verify the OpenMetadata server is running and check its version. The `/system/version` endpoint returns server metadata.

In [ ]:
# Check server version
resp = requests.get(f"{OM_URL}/system/version", headers=headers)
version_info = resp.json()

print("OpenMetadata Server Info:")
print(f"  Version: {version_info.get('version', 'N/A')}")
print(f"  Revision: {version_info.get('revision', 'N/A')}")
print(f"  Timestamp: {version_info.get('timestamp', 'N/A')}")

## Exercise 2: List Database Services

A **database service** in OpenMetadata represents a connection to a database system. When the lab starts fresh, there are no database services registered yet. Let's verify this.

In [ ]:
# List all registered database services
resp = requests.get(f"{OM_URL}/services/databaseServices", headers=headers)
services = resp.json().get("data", [])

print(f"Registered database services: {len(services)}")
for svc in services:
    print(f"  - {svc['name']} (type: {svc.get('serviceType', 'N/A')})")

if not services:
    print("\nNo database services found. Let's add one in Exercise 3!")

## Exercise 3: Add Lab PostgreSQL as a Database Service

This is a key catalog skill: **configuring a database connector** so the catalog can discover metadata.

### Step-by-step in the OpenMetadata UI:

1. Open the OpenMetadata UI at **http://localhost:18585**
2. Log in with **admin / admin**
3. Navigate to **Settings** (gear icon) > **Services** > **Databases**
4. Click **Add New Service**
5. Select **Postgres** as the service type
6. Enter the following connection details:

| Field | Value | Notes |
|-------|-------|-------|
| Service Name | `ecommerce_postgres` | Descriptive name for the catalog |
| Host | `postgresql` | Docker service name (internal network) |
| Port | `5432` | Internal PostgreSQL port |
| Username | `lab` | The lab user created by init script |
| Password | `lab_password` | From .env.example |
| Database | `ecommerce` | The e-commerce sample database |

7. Click **Test Connection** to verify connectivity
8. Click **Save** to register the service
9. After saving, click **Add Ingestion** > **Add Metadata Ingestion**
10. Use default settings and click **Submit** to run the metadata ingestion

**Wait 1-2 minutes** for the ingestion to discover tables, then proceed to Exercise 4.

> **Note:** If the "Add Ingestion" option is not available (it requires the Airflow service which is excluded from this lab for memory reasons), you can trigger metadata ingestion via the API in the cell below.

In [ ]:
# Alternative: Create the database service via API
# Use this if you prefer the API approach, or if UI ingestion is unavailable

service_payload = {
    "name": "ecommerce_postgres",
    "serviceType": "Postgres",
    "connection": {
        "config": {
            "type": "Postgres",
            "hostPort": "postgresql:5432",
            "username": "lab",
            "authType": {
                "password": "lab_password"
            },
            "database": "ecommerce"
        }
    }
}

# Check if service already exists
check = requests.get(f"{OM_URL}/services/databaseServices/name/ecommerce_postgres", headers=headers)
if check.status_code == 200:
    print("Service 'ecommerce_postgres' already exists.")
    service_id = check.json()["id"]
else:
    resp = requests.post(f"{OM_URL}/services/databaseServices", headers=headers, json=service_payload)
    if resp.status_code in (200, 201):
        service_id = resp.json()["id"]
        print(f"Created service 'ecommerce_postgres' (id: {service_id})")
    else:
        print(f"Error creating service: {resp.status_code} - {resp.text}")
        service_id = None

if service_id:
    print(f"\nService ID: {service_id}")
    print("Now use the UI to add a metadata ingestion, or proceed to Exercise 4.")

## Exercise 4: List Discovered Tables

After the database service is registered and metadata ingestion has run, OpenMetadata discovers all tables. Let's list them.

> **Note:** If you just created the service, you may need to wait 1-2 minutes for ingestion to complete, or manually run ingestion from the UI. If tables are not found, re-run this cell after ingestion.

In [ ]:
# List all tables in the catalog
resp = requests.get(
    f"{OM_URL}/tables",
    headers=headers,
    params={"limit": 50, "fields": "columns,tags"}
)
tables = resp.json().get("data", [])

print(f"Discovered tables: {len(tables)}")
print("=" * 60)
for table in tables:
    fqn = table.get("fullyQualifiedName", "N/A")
    col_count = len(table.get("columns", []))
    desc = table.get("description", "No description")
    print(f"  {fqn}")
    print(f"    Columns: {col_count} | Description: {desc[:60]}")
    print()

## Exercise 5: Inspect Column Metadata

Let's look at the detailed column metadata for the `customers` table. This shows how OpenMetadata captures column names, data types, and constraints.

In [ ]:
# Get detailed metadata for the customers table
# The FQN (Fully Qualified Name) follows the pattern:
# <service>.<database>.<schema>.<table>
TABLE_FQN = "ecommerce_postgres.ecommerce.public.customers"

resp = requests.get(
    f"{OM_URL}/tables/name/{TABLE_FQN}",
    headers=headers,
    params={"fields": "columns,tags,owner,tableConstraints"}
)

if resp.status_code == 200:
    table = resp.json()
    print(f"Table: {table['fullyQualifiedName']}")
    print(f"Description: {table.get('description', 'None')}")
    print(f"Table Type: {table.get('tableType', 'N/A')}")
    print()
    
    print("Columns:")
    print("-" * 70)
    print(f"{'Name':<20} {'Type':<15} {'Nullable':<10} {'Description'}")
    print("-" * 70)
    for col in table.get("columns", []):
        name = col.get("name", "")
        dtype = col.get("dataType", "")
        nullable = "YES" if col.get("constraint") != "NOT_NULL" else "NO"
        desc = col.get("description", "-")[:30]
        print(f"  {name:<20} {dtype:<15} {nullable:<10} {desc}")
else:
    print(f"Table not found: {resp.status_code}")
    print("Make sure you've added the ecommerce_postgres service and run metadata ingestion.")
    print("\nTip: Check Exercise 3 instructions for setting up the connector.")

## Summary

In this notebook, you learned:

1. **Server health:** How to check OpenMetadata server status via the REST API
2. **Database services:** How services represent connections to database systems
3. **Connector setup:** How to register a PostgreSQL database in the catalog (a core catalog administration skill)
4. **Table discovery:** How the catalog automatically discovers tables after ingestion
5. **Column metadata:** How to inspect column-level details (types, constraints, descriptions)

**Key concept:** A data catalog is only as useful as the metadata it contains. The first step is always connecting data sources and running discovery.

**Next:** [02-catalog-api-basics.ipynb](02-catalog-api-basics.ipynb) - Learn CRUD operations on catalog metadata via the REST API.